# Part 1 — Scope, Ownership, and Inputs

### Objective

Use validation data to select the operational model, horizon, stable monitoring window, threshold, and alert policy; then evaluate the fully locked system once on test patients.

### Inputs

Versioned snapshot data, fixed patient splits, locked candidate pipelines, hourly predictions, and event-driven replay predictions. Urine-output robustness labels are included when enabled; trajectory outputs are out of scope.

### Status

Run 2026-11-10: the locked test evaluation (`configs/evaluation_test_v1.yaml`) and the retraining sensitivity analyses (`configs/robustness_retrain_v1.yaml`) are complete; the cells below show their aggregate outputs. Parts 2-9 summarise the validation-stage work of notebooks 02, Demo I, KeJin and Jiaju.


In [1]:
# Notebook 03 entry point. The locked test evaluation runs in src/aki_final/test_evaluation.py and the retraining
# sensitivity analyses in src/aki_robustness/retrain.py; both freeze their protocol (config, input and code hashes)
# before reading outcomes. This notebook shows their aggregate outputs and, on request, re-runs them.
# Only aggregate tables are printed; no patient-level record is displayed.
import json
import sys
from pathlib import Path

import pandas as pd
import yaml


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for p in [start, *start.parents]:
        if (p / "configs" / "evaluation_test_v1.yaml").exists() and (p / "pipeline").exists():
            return p
    raise FileNotFoundError("Run this notebook from inside the AIForHealth-AKIEarlyWarning repository.")


ROOT = find_repo_root()
sys.path.insert(0, str(ROOT / "src"))
TEST = ROOT / "artifacts" / "reports" / "test_v1"
RETRAIN = ROOT / "artifacts" / "reports" / "robustness_retrain_v1"
POLICY_RUN = ROOT / "artifacts" / "handoff" / "policy_v3" / "validation-policy-v3-66054ed84c02e348"
JIAJU = ROOT / "artifacts" / "handoff" / "jiaju"
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

# Re-running is optional. Both studies reuse their frozen prespec and refuse changed code or inputs, so a re-run can
# only reproduce the same numbers; test results are never used to revise the locked system.
RERUN_TEST = False          # about 1 h; needs the local dataset, MIMIC LABEVENTS and the model artifacts
RERUN_RETRAIN = False       # about 2-3 h; retrains 7 LightGBM variants x 4 horizons (cached heads are reused)
if RERUN_TEST:
    from aki_final.test_evaluation import LockedEvaluation
    ev = LockedEvaluation(ROOT)
    for step in [ev.snapshot_table, ev.alert_table, ev.subgroup_table, ev.robustness_table, ev.shap_table,
                 ev.event_replay_table]:
        step()
    ev.run_manifest()
if RERUN_RETRAIN:
    from aki_robustness.retrain import RetrainStudy
    RetrainStudy(ROOT).run()

manifest = json.loads((TEST / "run_manifest.json").read_text(encoding="utf-8"))
print("Locked test evaluation:", manifest["version"], "| split", manifest["split"], "| patients", manifest["patients"],
      "| hourly origins", manifest["origins"])
print("Prespec sha256:", manifest["prespec_sha256"], "| finished", manifest["finished_at"][:19])

Locked test evaluation: test-evaluation-v1 | split test | patients 4959 | hourly origins 202797
Prespec sha256: 14ab07ff07ad5d469745a85e35eab658685fcd16db67c830cc098850e363ee53 | finished 2026-11-10T03:42:51


# Part 2 — Validate Evaluation Artifacts

### Objective

Confirm that all evaluation inputs are complete and compatible.

### Checks

- Unique model, horizon, and snapshot keys
- Probability bounds
- Required schemas and versions
- Missing predictions
- Patient split disjointness using patient_id
- One-to-one patient_id to selected icustay_id mapping
- Chronological ordering
- Matching label and model horizon definitions

Stop evaluation when artifact integrity fails.


In [2]:
# Artifact integrity: checks A1-A12 (src/aki_eval/checks.py, run on validation in KeJin_alert_policy_and_checks.ipynb)
# and the notebook-03 relabel check (test labels recomputed from onset times match the stored labels).
checks = pd.read_csv(POLICY_RUN / "checks.csv")
print(checks[["id", "check", "passed"]].to_string(index=False))
assert checks["passed"].all(), "an artifact check failed"
relabel = json.loads((TEST / "relabel_check.json").read_text(encoding="utf-8"))
print("\nTest primary-label mismatches by horizon:", relabel["primary_label_mismatches_by_horizon"])

 id                                         check  passed
 A1               Unique prediction/snapshot keys    True
 A2                 Finite probabilities in [0,1]    True
 A3         Schemas, versions and immutable model    True
 A4      Every eligible snapshot has each horizon    True
 A5                         Frozen patient splits    True
 A6                 One selected stay per patient    True
 A7             Chronology and ICU-relative hours    True
 A8     Horizon labels, eligibility and alignment    True
 A9        No at/after-onset prediction snapshots    True
A10 Past-only feature parity and future injection    True
A11                  No future urine contribution    True
A12    No test use declared and OOF patient folds    True

Test primary-label mismatches by horizon: {'6': 0, '12': 0, '24': 0, '48': 0}


# Part 3 — Run Leakage and Outcome Tests

### Objective

Audit temporal validity before reporting performance.

### Required tests

- Every feature event occurs at or before prediction time.
- Every positive label occurs inside its horizon.
- Snapshots at or after AKI onset are excluded.
- Post-snapshot urine is absent from predictors.
- Preprocessing, weighting, selection, and calibration did not use test information.
- No patient_id appears in multiple splits.


In [3]:
# Leakage and outcome tests (A5-A12): split disjointness, one stay per patient, chronology, label windows, no origin at
# or after onset, past-only feature replay with future-event injection, no post-snapshot urine, no test use in fitting.
leak = checks[checks["id"].isin(["A5", "A6", "A7", "A8", "A9", "A10", "A11", "A12"])]
print(leak[["id", "check", "passed", "detail"]].to_string(index=False, max_colwidth=90))

 id                                         check  passed                                                                                     detail
 A5                         Frozen patient splits    True                                            patients=35732; mismatched_or_unassigned_rows=0
 A6                 One selected stay per patient    True                                                                 mismatched_identity_rows=0
 A7             Chronology and ICU-relative hours    True                                                                  invalid_chronology_rows=0
 A8     Horizon labels, eligibility and alignment    True                                invalid_snapshot_labels=0; inconsistent_prediction_labels=0
 A9        No at/after-onset prediction snapshots    True                                                                   at_or_after_onset_rows=0
A10 Past-only feature parity and future injection    True sampled_rows=2048; compared_features=246; replay

# Part 4 — Compare Models and Horizons on Validation

### Objective

Compare LR, RF, and boosted trees at 6h, 12h, 24h, and 48h using validation data, while accounting for changing outcome prevalence and follow-up eligibility across hours_since_icu.

### Metrics

AUROC, AUPRC, sensitivity, specificity, F1, Brier score, calibration, and prevalence, overall and for each horizon and prespecified hours_since_icu segment.

### Rule

Validation results guide selection and are not the final unbiased performance claim.


In [4]:
# Validation comparison of model families and horizons (notebook 02). The table lives with the local model artifacts;
# the same numbers are listed in the README.
cal = ROOT / "artifacts" / "models" / "aki_ml_v1_28e0b1fb83" / "reports" / "calibration_summary.csv"
if cal.exists():
    t = pd.read_csv(cal).query("set == 'validation' and calibration == 'platt'")
    print("Validation AUROC (Platt):")
    print(t.pivot_table(index=["family", "feature_set"], columns="horizon_h", values="auroc").round(3).to_string())
else:
    print("Local model reports not present; see README 'Current status' for the validation AUROC table.")

Validation AUROC (Platt):
horizon_h                6      12     24     48
family   feature_set                            
lightgbm full         0.862  0.837  0.806  0.784
         no_dc        0.865  0.839  0.809  0.782
lr       full         0.818  0.800  0.779  0.755
rf       full         0.842  0.816  0.785  0.765
xgboost  full         0.858  0.836  0.805  0.782


# Part 5 — Inspect Calibration and Predictor Stability

### Objective

Check whether risk estimates and predictors remain stable across ICU time and horizons.

### Analyses

- Calibration by hours_since_icu and horizon
- Coefficient, importance, or SHAP stability
- Feature-distribution and missingness shifts
- Risk-score volatility between updates
- Monotonic consistency across 6h, 12h, 24h, and 48h risk

### Reliability assessment

Define a reliable early prediction as one that has acceptable validation discrimination and calibration, provides clinically useful warning lead time, and stays within a prespecified false-alert burden. Apply the same criteria across candidate horizons and hours_since_icu segments; do not call a model reliable from AUROC alone.

Interpret predictors as associations, not causes.


In [5]:
# Calibration and predictor stability on validation (pipeline/Jiaju_stability.ipynb).
rel = pd.read_csv(JIAJU / "reliability_summary.csv")
print("Horizon x ICU-segment cells meeting every prespecified reliability criterion:",
      int(rel["prespecified_all_pass"].sum()), "of", len(rel))
mono = pd.read_csv(JIAJU / "horizon_monotonicity.csv").query("segment == 'all' and calibration == 'platt'")
print("\nShare of origins where the shorter horizon has the higher risk:")
print(mono[["horizon_h", "next_horizon_h", "violation_fraction", "excess_p95"]].round(4).to_string(index=False))

Horizon x ICU-segment cells meeting every prespecified reliability criterion: 22 of 36

Share of origins where the shorter horizon has the higher risk:
 horizon_h  next_horizon_h  violation_fraction  excess_p95
         6              12              0.0438      0.0380
        12              24              0.0165      0.0675
        24              48              0.0083      0.0925


# Part 6 — Select the Stable Monitoring Window

### Objective

Choose the ICU period in which prediction is accurate, calibrated, early enough, and operationally useful under the reliability criteria.

### Candidate segments

Use a limited prespecified grid such as 8–12, 12–18, 18–24, 24–36, and 36–48 hours.

### Criteria

Discrimination, calibration, sample size, prevalence, risk stability, potential lead time, and false-alert burden.

Lock the monitoring start and end times before test evaluation.


In [6]:
# Stable monitoring window: chosen in Demo I on training out-of-fold predictions and confirmed on validation.
policy = json.loads((ROOT / "artifacts" / "policy" / "policy_v3.json").read_text(encoding="utf-8"))
print("Monitoring window:", policy["threshold"]["monitoring_window"])

Monitoring window: ICU hours 1-72


# Part 7 — Select the Operational Horizon and Threshold

### Objective

Choose whether the operational alert should use 6h, 12h, 24h, 48h, or a prespecified combination, then select its risk threshold using reliability and class-imbalance-aware validation criteria.

### Guidance

- 48h supports early screening but may create more uncertainty.
- 24h is a candidate primary operational warning horizon.
- 6h and 12h represent higher short-term urgency.
- Compare horizons only among their eligible risk sets and report the corresponding positive prevalence; do not interpret raw accuracy across horizons with different follow-up or prevalence.

### Rule

Evaluate a limited threshold grid on saved validation predictions; threshold testing never retrains the model.


In [7]:
# Operational horizon and threshold (locked policy v3).
th = policy["threshold"]
print(f"Alert horizon 24 h, threshold {th['value']} ({th['comparison']}), locked={th['locked']}")
print("Policy version:", policy["policy_version"])

Alert horizon 24 h, threshold 0.21 (>=), locked=True
Policy version: validation-policy-v3


# Part 8 — Select the Alert Policy on Validation

### Objective

Choose the complete warning logic.

### Candidate components

- Threshold crossing
- Sustained high risk
- Rapid risk increase
- Multi-horizon escalation
- Early-stage stricter confidence requirement
- Alert cooldown and repeat-alert suppression

Select every policy parameter on validation data and define exactly when alerts start, end, and repeat.


In [8]:
# Alert policy selected on 70% of validation patients and confirmed on the other 30% (docs/ALERT-POLICY.md).
choice = json.loads((POLICY_RUN / "choice.json").read_text(encoding="utf-8"))["chosen"]
print({k: choice[k] for k in ["candidate", "threshold", "cooldown_hours", "confirm_horizon_h", "confirm_ratio"]})
conf = pd.read_csv(POLICY_RUN / "policy_confirmation_validation.csv")
keep = ["false_alerts_per_patient_day", "alert_ppv", "event_detection_rate", "lead_time_median_h", "non_aki_patients_alerted"]
print(conf[conf["metric"].isin(keep)].pivot(index="metric", columns="comparison", values="estimate").round(3).to_string())
print("\nOwner decision:", policy.get("owner_decision"))

{'candidate': 'confirm12_cool12', 'threshold': 0.21, 'cooldown_hours': 12, 'confirm_horizon_h': 12.0, 'confirm_ratio': 0.5}
comparison                    baseline  candidate  difference
metric                                                       
alert_ppv                        0.337      0.339       0.002
event_detection_rate             0.769      0.719      -0.051
false_alerts_per_patient_day     0.424      0.230      -0.194
lead_time_median_h              12.806     11.663      -1.143
non_aki_patients_alerted         0.358      0.311      -0.047

Owner decision: {'date': '2026-11-09', 'decision': 'accept the v3 trade-off despite the missed detection-loss criterion', 'failed_criterion': 'event_detection_rate >= 0.7195 (observed 0.7186 of 295 events)'}


# Part 9 — Lock the Evaluation Protocol

### Objective

Freeze the model, horizon, primary Platt calibration, isotonic calibration sensitivity analysis, monitoring window, threshold, alert logic, reliability criteria, class-imbalance strategy, subgroup definitions, sensitivity analyses, and metrics.

### Rule

After locking, test results must not be used to revise the system. Any revision begins a new labeled development cycle and requires untouched evaluation data.


In [9]:
# Locked evaluation protocol, frozen before any test outcome was read.
protocol = yaml.safe_load((ROOT / "configs" / "evaluation_test_v1.yaml").read_text(encoding="utf-8"))
prespec = json.loads((TEST / "prespec.json").read_text(encoding="utf-8"))
print("Protocol:", protocol["version"], "| policy", protocol["policy"], "| comparator", protocol["comparator_policy"])
print("Frozen at", prespec["created_at"][:19], "| hashed code files:", len(prespec["specification"]["source_sha256"]))

Protocol: test-evaluation-v1 | policy artifacts/policy/policy_v3.json | comparator artifacts/demo_i/policy.json
Frozen at 2026-11-10T02:59:58 | hashed code files: 16


# Part 10 — Perform Final Snapshot-Level Test Evaluation

### Objective

Evaluate the frozen system on test snapshots.

### Outputs

- AUROC and AUPRC
- Sensitivity, specificity, and F1
- Brier score and calibration plots
- Performance by ICU time and horizon
- Results for 6h, 12h, 24h, and 48h horizons
- Patient-clustered uncertainty intervals

Do not treat correlated hourly rows as independent observations when estimating uncertainty.


In [10]:
# Snapshot-level test results, ICU hours 1-72, Platt calibration, patient-bootstrap 95% CI.
snap = pd.read_csv(TEST / "snapshot_metrics.csv")
overall = snap.query("scope == 'hours 1-72' and calibration == 'platt'")
cols = ["horizon_h", "rows", "positives", "prevalence", "auroc", "auroc_ci_low", "auroc_ci_high", "auprc", "brier",
        "observed_over_expected", "calibration_slope"]
print(overall[cols].round(3).to_string(index=False))
segments = [f"{a}-{b}h" for a, b in protocol["segments"]]
seg = snap.query("scope != 'hours 1-72' and calibration == 'platt'")
print("\nAUROC by ICU-hour segment:")
print(seg.pivot(index="scope", columns="horizon_h", values="auroc").reindex(segments).round(3).to_string())
print("\n24 h point warning (eligible rows):")
print(pd.read_csv(TEST / "point_warning_24h.csv").round(3).to_string(index=False))

 horizon_h   rows  positives  prevalence  auroc  auroc_ci_low  auroc_ci_high  auprc  brier  observed_over_expected  calibration_slope
         6 185059       5191       0.028  0.866         0.856          0.876  0.184  0.025                   0.981              1.003
        12 166998       9630       0.058  0.841         0.830          0.853  0.285  0.047                   0.984              1.036
        24 134149      15987       0.119  0.817         0.803          0.832  0.419  0.086                   0.978              1.063
        48  95052      23232       0.244  0.792         0.775          0.810  0.573  0.145                   0.982              1.069

AUROC by ICU-hour segment:
horizon_h     6      12     24     48
scope                                
1-3h       0.843  0.815  0.764  0.732
4-6h       0.858  0.823  0.791  0.760
7-12h      0.849  0.839  0.811  0.784
13-18h     0.888  0.853  0.822  0.793
19-24h     0.873  0.824  0.824  0.794
25-36h     0.860  0.854  0.827  0.81

# Part 11 — Perform Patient-Level and AKI-Event-Level Evaluation

### Objective

Measure whether the warning system helps at the level that matters operationally.

### Metrics

- Proportion of future AKI events detected
- First-warning lead time per event
- Patients alerted without future AKI
- False alerts per patient-day
- Repeat-alert burden
- Time from eligibility to first alert

Define how multiple alerts map to one AKI event before computing results.


In [11]:
# Patient- and AKI-event-level alerting on test: policy v3 versus demo-v1 on the same risks.
alerts = pd.read_csv(TEST / "alert_metrics.csv")
table = alerts.pivot(index="metric", columns="policy", values="estimate")
ci = alerts.query("policy == 'policy_v3'").set_index("metric")[["ci_low", "ci_high"]].add_prefix("v3_")
print(table.join(ci).round(3).to_string())

                                    demo_v1  difference_v3_minus_demo_v1  policy_v3  v3_ci_low  v3_ci_high
metric                                                                                                    
aki_events                          959.000                        0.000    959.000    905.000    1011.050
alert_ppv                             0.340                       -0.005      0.335      0.315       0.356
alerts_per_alerted_patient            3.007                       -1.264      1.743      1.701       1.785
alerts_per_patient_day                0.798                       -0.380      0.418      0.401       0.436
event_detection_rate                  0.766                       -0.049      0.717      0.688       0.745
false_alerts_per_patient_day          0.415                       -0.188      0.227      0.215       0.239
hours_to_first_alert_median           1.000                        0.000      1.000      1.000       1.000
lead_time_mean_h                     

# Part 12 — Evaluate Event-Driven Replay

### Objective

Simulate dashboard operation by recomputing risk whenever a relevant EHR variable changes.

### Checks

- Compare event-driven and hourly risk distributions.
- Measure latency, duplicate predictions, and alert burden.
- Apply an operational debounce or cooldown without hiding clinically meaningful changes.
- Confirm that arbitrary event times do not create material training-deployment mismatch.


In [12]:
# Event-driven replay on test: risk recomputed at every input availability time.
disc = pd.read_csv(TEST / "event_replay_discrimination.csv")
print(disc[["horizon_h", "grid", "rows", "auroc", "brier", "observed_over_expected"]].round(3).to_string(index=False))
er = pd.read_csv(TEST / "event_replay_alerts.csv")
print()
print(er[["grid", "alerts_per_patient_day", "false_alerts_per_patient_day", "alert_ppv", "event_detection_rate",
          "lead_time_median_h"]].round(3).to_string(index=False))
print(json.loads((TEST / "event_replay_summary.json").read_text(encoding="utf-8")))

 horizon_h         grid   rows  auroc  brier  observed_over_expected
        12 event-driven 414001  0.846  0.057                   1.052
        12       hourly 166998  0.841  0.047                   0.984
        24 event-driven 341669  0.829  0.098                   1.051
        24       hourly 134149  0.817  0.086                   0.978

        grid  alerts_per_patient_day  false_alerts_per_patient_day  alert_ppv  event_detection_rate  lead_time_median_h
event-driven                   0.431                         0.238      0.328                 0.734              11.992
      hourly                   0.418                         0.227      0.335                 0.717              11.637
{'event_times': 478698, 'updates_per_patient_median': 73.0, 'abs_risk_change_between_updates_median': 0.0018796548247337341, 'abs_risk_change_between_updates_p95': 0.030989676713943478}


# Part 13 — Defer Trajectory and Combined-Label Extensions

### Objective

Do not evaluate future trajectories or a combined-label model in the current notebook set.

### Scope boundary

Complete the creatinine-only classification and urine-output robustness analysis first. Any combined-label training or trajectory prediction must be defined in a new notebook and a new experiment version.

### Future work

If pursued later, specify separate targets, eligibility rules, metrics, and leakage tests before implementation.


In [13]:
print("Deferred by design: no trajectory model and no combined creatinine + urine-output label in notebook 03.")

Deferred by design: no trajectory model and no combined creatinine + urine-output label in notebook 03.


# Part 14 — Compute Interpretation Outputs

### Objective

Explain global model behavior and individual current-risk predictions.

### Outputs

Global feature importance, LR coefficients where applicable, SHAP summaries, patient-level top drivers, feature values, and data recency.

### Rules

Keep patient-level explanations local, verify the explained probability scale, and avoid causal claims.


In [14]:
# Global TreeSHAP on 1,500 test patients (mean |SHAP| on the raw-score scale; associations, not causes).
shap = pd.read_csv(TEST / "shap_top20.csv")
print(shap.query("horizon_h == 24").head(10)[["rank", "feature", "mean_abs_shap", "share_of_total"]].round(4)
      .to_string(index=False))

 rank                      feature  mean_abs_shap  share_of_total
    1 creatinine_rise_from_48h_min         0.1762          0.0443
    2              creatinine_last         0.1385          0.0348
    3              hours_since_icu         0.1340          0.0337
    4                  ptt_max_48h         0.1092          0.0274
    5                    age_years         0.0862          0.0217
    6          ph_hours_since_last         0.0833          0.0209
    7   creatinine_ratio_to_7d_min         0.0786          0.0197
    8             bicarbonate_last         0.0717          0.0180
    9                sbp_slope_24h         0.0714          0.0179
   10             urine_ml_kg_h_6h         0.0707          0.0178


# Part 15 — Perform Subgroup, System-Era, and Sensitivity Analyses

### Objective

Test whether conclusions depend on population, source system, or study definitions.

### Analyses

- Age, sex, and ICU type
- CareVue versus MetaVision robustness
- Alternative baseline-creatinine definitions
- Creatinine-only primary endpoint versus the separate urine-output KDIGO 2012 robustness endpoint
- Urine-output weight source, rolling-window, missing-output, and renal replacement therapy sensitivity
- Feature-window and missingness strategies
- Snapshot weighting and alert-policy variants

### Temporal validation note

Do not order patients by shifted MIMIC-III calendar year. Use valid system-era provenance or a later approved external database for true temporal validation.


In [15]:
# Subgroups, system era, alternative endpoint and baselines (test), then the retraining sensitivity analyses.
sub = pd.read_csv(TEST / "subgroups_snapshot.csv")
print(sub[["variable", "level", "patients", "auroc_24h", "auroc_24h_ci_low", "auroc_24h_ci_high",
           "observed_over_expected_24h"]].round(3).to_string(index=False))
sa = pd.read_csv(TEST / "subgroups_alerts.csv")
print("\nEvent detection by subgroup (policy v3):")
print(sa.query("metric == 'event_detection_rate'")[["variable", "level", "estimate", "ci_low", "ci_high"]].round(3)
      .to_string(index=False))
rob = pd.read_csv(TEST / "robustness_snapshot.csv").query("horizon_h == 24")
print("\nRobustness, 24 h:")
print(rob[["analysis", "calibration", "rows", "auroc", "brier", "observed_over_expected"]].round(3).to_string(index=False))
ra = pd.read_csv(TEST / "robustness_alerts.csv")
print(ra[["analysis", "event_detection_rate", "alert_ppv", "false_alerts_per_patient_day"]].round(3).to_string(index=False))

# Feature-window and missingness variants, each retrained with the notebook-02 procedure (configs/robustness_retrain_v1.yaml).
rm = pd.read_csv(RETRAIN / "snapshot_metrics.csv")
print("\nRetrained variants (stored notebook-01 features), AUROC:")
print(rm.pivot_table(index="variant", columns=["split", "horizon_h"], values="auroc").round(3).to_string())
d = pd.read_csv(RETRAIN / "test_differences.csv").query("metric == 'auroc_difference'")
print("\nTest AUROC minus the locked model (paired patient bootstrap, 95% CI), 24 h:")
print(d.query("horizon_h == 24")[["variant", "estimate", "ci_low", "ci_high"]].round(4).to_string(index=False))
print("\nHarness check (reference_refit reproduces the locked model):",
      json.loads((RETRAIN / "harness_check.json").read_text(encoding="utf-8"))["passed"])

      variable      level  patients  auroc_24h  auroc_24h_ci_low  auroc_24h_ci_high  observed_over_expected_24h
     age_years      18-64      2398      0.815             0.791              0.836                       0.962
     age_years      65-79      1491      0.816             0.794              0.837                       0.998
     age_years        80+      1070      0.806             0.778              0.834                       0.973
           sex       male      2764      0.814             0.797              0.831                       1.005
           sex     female      2195      0.821             0.800              0.841                       0.943
first_careunit       MICU      1685      0.821             0.796              0.847                       0.881
first_careunit       SICU       787      0.822             0.776              0.866                       0.841
first_careunit       CSRU      1056      0.761             0.737              0.787                     


Event detection by subgroup (policy v3):
      variable      level  estimate  ci_low  ci_high
     age_years      18-64     0.663   0.618    0.711
     age_years      65-79     0.760   0.712    0.802
     age_years        80+     0.742   0.690    0.801
           sex       male     0.713   0.678    0.753
           sex     female     0.723   0.680    0.764
first_careunit       MICU     0.686   0.629    0.740
first_careunit       SICU     0.683   0.598    0.768
first_careunit       CSRU     0.805   0.767    0.846
first_careunit        CCU     0.671   0.605    0.744
first_careunit      TSICU     0.614   0.517    0.724
 source_system    carevue     0.695   0.655    0.730
 source_system metavision     0.747   0.705    0.787

Robustness, 24 h:
                   analysis calibration   rows  auroc  brier  observed_over_expected
 primary (creatinine KDIGO)       platt 134149  0.817  0.086                   0.978
 primary (creatinine KDIGO)    isotonic 134149  0.817  0.086                   0

                   analysis  event_detection_rate  alert_ppv  false_alerts_per_patient_day
 primary (creatinine KDIGO)                 0.717      0.335                         0.227
urine-output KDIGO endpoint                 0.388      0.638                         0.119
  baseline: mdrd_if_missing                 0.663      0.326                         0.187
 baseline: min_with_history                 0.720      0.339                         0.224

Retrained variants (stored notebook-01 features), AUROC:
split                              test                      validation                     
horizon_h                            6      12     24     48         6      12     24     48
variant                                                                                     
missing_median_impute             0.865  0.839  0.817  0.793      0.862  0.837  0.809  0.782
missing_median_impute_indicators  0.867  0.839  0.817  0.791      0.864  0.838  0.807  0.782
missing_no_measurement


Harness check (reference_refit reproduces the locked model): True


# Part 16 — Export Dashboard Data and Final Report

### Objective

Create local event-driven dashboard inputs and aggregate research outputs.

### Dashboard views

Current multi-horizon AKI risk, warning status, top SHAP drivers, latest measurements, data recency, and patient timeline.

### Final report

Record locked definitions, model and policy versions, snapshot-, patient-, and event-level results, calibration, robustness, limitations, configuration, and Git commit.

Patient-level MIMIC-derived data remain local; shareable demonstrations must use synthetic data.


In [16]:
# Dashboard export: the public static demo replays four synthetic patients scored by the locked model and policy v3
# (scripts/export_static_demo.py -> dashboard/web/public/static-demo/demo.json; `pnpm build:static` -> demo-static/).
demo = json.loads((ROOT / "dashboard" / "web" / "public" / "static-demo" / "demo.json").read_text(encoding="utf-8"))
print("Static demo:", len(demo["patients"]), "synthetic patients,", len(demo["observations"]), "observations,",
      len(demo["predictions"]), "predictions | policy", demo["model"]["policy_version"])
print("Final report: docs/FINAL-REPORT.md | deployment: docs/STATIC-DEMO.md")

Static demo: 4 synthetic patients, 3052 observations, 1120 predictions | policy validation-policy-v3
Final report: docs/FINAL-REPORT.md | deployment: docs/STATIC-DEMO.md
